# Option A — Real Photo CNN (ASL Alphabet Dataset)
Trains MobileNetV2 on real webcam-style hand photos at 64×64 RGB.
Dataset: https://www.kaggle.com/datasets/grassknoted/asl-alphabet (87k images, 29 classes)

Output: `models/optionA.tflite` — drop into `bonus/webcam_trials/models/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install & Imports

In [ ]:
!pip install -q kaggle tensorflow

import os, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Download Dataset

In [ ]:
# Upload kaggle.json to Colab first, or set credentials below
if not os.path.exists('/root/.kaggle/kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.environ['KAGGLE_USERNAME'] = 'abdullahashiry'
    os.environ['KAGGLE_KEY']      = 'KGAT_331632d901a6cb7a05431b55135bd8c2'

if not os.path.exists('asl_alphabet_train'):
    !kaggle datasets download -d grassknoted/asl-alphabet --unzip

TRAIN_DIR = 'asl_alphabet_train/asl_alphabet_train'
print('Classes:', sorted(os.listdir(TRAIN_DIR)))

## 3. Config

In [ ]:
IMG_SIZE   = 64       # 64×64 RGB — 4× more info than Sign MNIST 28×28 gray
BATCH_SIZE = 64
EPOCHS     = 30
AUTOTUNE   = tf.data.AUTOTUNE

# Use A-Z only (exclude del, nothing, space) for ASL letter recognition
# J and Z are included — they appear as static poses in this dataset
ALL_CLASSES = sorted([c for c in os.listdir(TRAIN_DIR) if len(c) == 1 and c.isalpha()])
NUM_CLASSES = len(ALL_CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(ALL_CLASSES)}
print(f'Using {NUM_CLASSES} classes: {ALL_CLASSES}')

## 4. tf.data Pipeline with Augmentation

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label


augment_train = keras.Sequential([
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.12),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomFlip('horizontal'),      # mirror = same letter, different hand
    layers.RandomBrightness(0.15, value_range=(0, 1)),  # value_range=(0,1) for float images
    layers.RandomContrast(0.20),
], name='augment')


def augment_fn(img, label):
    img = augment_train(img, training=True)
    img = tf.clip_by_value(img, 0.0, 1.0)   # clamp after augmentation
    # small Gaussian noise
    if tf.random.uniform(()) < 0.3:
        img = tf.clip_by_value(
            img + tf.random.normal(tf.shape(img), stddev=0.02), 0.0, 1.0
        )
    return img, label


# Collect file paths and labels
all_paths, all_labels = [], []
for cls in ALL_CLASSES:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(CLASS_TO_IDX[cls])

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

# 90/10 split
idx      = np.random.permutation(len(all_paths))
split    = int(0.9 * len(idx))
tr_idx, val_idx = idx[:split], idx[split:]

def make_dataset(paths, labels, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if augment:
        ds = ds.shuffle(len(paths), seed=42)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train = make_dataset(all_paths[tr_idx],  all_labels[tr_idx],  augment=True)
ds_val   = make_dataset(all_paths[val_idx], all_labels[val_idx], augment=False)

print(f'Train: {len(tr_idx)} images  Val: {len(val_idx)} images')
print(f'Train batches: {len(ds_train)}  Val batches: {len(ds_val)}')

## 5. Sample Augmented Images

In [ ]:
imgs, labels = next(iter(ds_train))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].numpy())
    ax.set_title(ALL_CLASSES[labels[i]], fontsize=10)
    ax.axis('off')
plt.suptitle('Augmented training samples (real photos)')
plt.tight_layout()
plt.show()

## 6. Model — MobileNetV2 (transfer learning)

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
# Phase 1: freeze base, train head only
base.trainable = False

inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dropout(0.3)(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inp, out, name='OptionA_MobileNetV2')
model.summary()
print(f'Trainable params: {sum(p.numpy().size for p in model.trainable_variables):,}')

## 7. Phase 1 — Train head (10 epochs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

h1 = model.fit(ds_train, epochs=10, validation_data=ds_val,
               callbacks=[keras.callbacks.EarlyStopping(
                   monitor='val_accuracy', patience=4, restore_best_weights=True)])

## 8. Phase 2 — Fine-tune top 50 layers

In [ ]:
base.trainable = True
# Freeze all except last 50 layers
for layer in base.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),   # lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        'best_optionA.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6
    ),
]

h2 = model.fit(ds_train, epochs=EPOCHS, validation_data=ds_val, callbacks=callbacks)

## 9. Evaluate

In [ ]:
loss, acc = model.evaluate(ds_val, verbose=0)
print(f'Val accuracy: {acc*100:.2f}%')

all_h = {k: h1.history.get(k, []) + h2.history.get(k, []) for k in ['accuracy','val_accuracy','loss','val_loss']}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(all_h['accuracy'], label='train'); ax1.plot(all_h['val_accuracy'], label='val')
ax1.axvline(10, color='gray', linestyle='--', label='fine-tune start')
ax1.set_title('Accuracy'); ax1.legend()
ax2.plot(all_h['loss'], label='train'); ax2.plot(all_h['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.show()

## 10. Export TFLite + Save classes

In [ ]:
import json

# Save class list so Livestream.py knows the label mapping
with open('optionA_classes.json', 'w') as f:
    json.dump(ALL_CLASSES, f)
print('Classes:', ALL_CLASSES)

# Float32 TFLite
converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('optionA.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'FP32: {os.path.getsize("optionA.tflite")/1e6:.2f} MB')

## 11. Copy to Drive

In [ ]:
import shutil

OUT = '/content/drive/MyDrive/CV552_SignLanguage/tflite'
os.makedirs(OUT, exist_ok=True)

shutil.copy('optionA.tflite',        os.path.join(OUT, 'optionA.tflite'))
shutil.copy('optionA_classes.json',  os.path.join(OUT, 'optionA_classes.json'))
shutil.copy('best_optionA.keras',    os.path.join(OUT, 'optionA.keras'))
print(f'Saved to {OUT}')
print('\nNext: download optionA.tflite → bonus/webcam_trials/models/optionA.tflite')